In [1]:
!pip install -q google-cloud-discoveryengine google-cloud-aiplatform[evaluation] google-cloud-modelarmor ipytest

In [2]:
import google.auth
import requests
import logging
import datetime
import time
import ipytest
import pytest

import vertexai
from vertexai.generative_models import GenerativeModel
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate
from google.cloud import discoveryengine_v1 as discoveryengine
from google.api_core.exceptions import AlreadyExists
import pandas as pd

ipytest.autoconfig()

### Config

In [3]:
credentials, PROJECT_ID = google.auth.default()

LOCATION       = "global"
DATA_STORE_ID  = "alaska-dept-of-snow"
GCS_BUCKET     = "gs://labs.roitraining.com/alaska-dept-of-snow"
GEMINI_MODEL   = "gemini-2.5-flash"
GEMINI_REGION  = "us-central1"

print(f"Project ID    : {PROJECT_ID}")
print(f"Data Store ID : {DATA_STORE_ID}")
print(f"GCS Bucket    : {GCS_BUCKET}")

Project ID    : qwiklabs-gcp-04-a13cec945fb9
Data Store ID : alaska-dept-of-snow
GCS Bucket    : gs://labs.roitraining.com/alaska-dept-of-snow


In [4]:
# ── DATA STORE SETUP ────────────────────────────────────────────

def create_data_store():
    client = discoveryengine.DataStoreServiceClient()
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    data_store = discoveryengine.DataStore(
        display_name="Alaska Dept of Snow FAQ",
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
    )

    try:
        operation = client.create_data_store(
            parent=parent,
            data_store=data_store,
            data_store_id=DATA_STORE_ID,
        )
        print("Creating data store — waiting for operation to complete...")
        result = operation.result(timeout=120)
        print(f"Data store created: {result.name}")
    except AlreadyExists:
        print(f"Data store '{DATA_STORE_ID}' already exists — skipping creation")


def import_documents():
    client = discoveryengine.DocumentServiceClient()
    parent = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"
        f"/dataStores/{DATA_STORE_ID}/branches/default_branch"
    )

    request = discoveryengine.ImportDocumentsRequest(
        parent=parent,
        gcs_source=discoveryengine.GcsSource(
            input_uris=[f"{GCS_BUCKET}/*"],
            data_schema="content",
        ),
        reconciliation_mode=discoveryengine.ImportDocumentsRequest.ReconciliationMode.INCREMENTAL,
    )

    print(f"Importing documents from {GCS_BUCKET}...")
    operation = client.import_documents(request=request)
    print(f"Import started — operation: {operation.operation.name}")
    return operation


def wait_for_import(operation, poll_interval=30, max_wait=600):
    print("Polling import status...")
    elapsed = 0
    while not operation.done():
        print(f"  Still indexing... ({elapsed}s elapsed)")
        time.sleep(poll_interval)
        elapsed += poll_interval
        if elapsed >= max_wait:
            print("Max wait reached — check AI Applications UI for status")
            return
    if operation.exception():
        print(f"Import failed: {operation.exception()}")
    else:
        print(f"Import complete: {operation.result()}")


create_data_store()
import_op = import_documents()
wait_for_import(import_op)

Data store 'alaska-dept-of-snow' already exists — skipping creation
Importing documents from gs://labs.roitraining.com/alaska-dept-of-snow...
Import started — operation: projects/131669982938/locations/global/collections/default_collection/dataStores/alaska-dept-of-snow/branches/0/operations/import-documents-12843410092380583924
Polling import status...
  Still indexing... (0s elapsed)
Import complete: error_samples {
  code: 3
  message: "File extension type is DS_Store, and it is not supported. Currently supported extensions are pdf, html, docx, pptx, xlsx and txt."
  details {
    type_url: "type.googleapis.com/google.rpc.ResourceInfo"
    value: "\0229gs://labs.roitraining.com/alaska-dept-of-snow/.DS_Store:0"
  }
}
error_samples {
  code: 3
  message: "File extension type is csv, and it is not supported. Currently supported extensions are pdf, html, docx, pptx, xlsx and txt."
  details {
    type_url: "type.googleapis.com/google.rpc.ResourceInfo"
    value: "\022Lgs://labs.roitrain

Note - the import errors above are ok - those are the csv version of all the txt file questions I imported and a mac metadata file we don't want and would expect an error for

In [5]:
def verify_data_store():
    client = discoveryengine.SearchServiceClient()
    serving_config = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"
        f"/dataStores/{DATA_STORE_ID}/servingConfigs/default_config"
    )

    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query="snow removal",
        page_size=3,
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
                return_snippet=True,
                max_snippet_count=2,
            ),
        ),
    )

    response = client.search(request)
    results = list(response.results)
    print(f"Verification search returned {len(results)} results\n")

    for i, result in enumerate(results, 1):
        doc = result.document
        print(f"--- Result {i} ---")
        print(f"  ID   : {doc.id}")
        if doc.derived_struct_data:
            print(f"  Title: {doc.derived_struct_data.get('title', 'N/A')}")
            print(f"  Link : {doc.derived_struct_data.get('link', 'N/A')}")
            for s in doc.derived_struct_data.get("snippets", []):
                if s.get("snippet"):
                    print(f"  Snippet: {s['snippet'][:200]}")
        print()

    if results:
        print("✓ Data store is ready")
    else:
        print("✗ No results yet — wait a few minutes and re-run this cell")

verify_data_store()

Verification search returned 3 results

--- Result 1 ---
  ID   : b00e395bfbc994e94bc151fd534c4812
  Title: faq-22
  Link : gs://labs.roitraining.com/alaska-dept-of-snow/faq-22.txt
  Snippet: ADS does not provide direct financial assistance. However, some state grants may be available to local governments for purchasing <b>snow removal</b> equipment.

--- Result 2 ---
  ID   : 873899085adc089def683b43405d09f2
  Title: faq-23
  Link : gs://labs.roitraining.com/alaska-dept-of-snow/faq-23.txt
  Snippet: Are ADS plows available for hire for private property? No. ADS resources are dedicated to public roads and infrastructure. Private <b>snow removal</b> must be&nbsp;...

--- Result 3 ---
  ID   : afcb46699d9f36182a006d8510cae1bd
  Title: faq-28
  Link : gs://labs.roitraining.com/alaska-dept-of-snow/faq-28.txt
  Snippet: How does ADS measure the success of its <b>snow removal</b> efforts? Key performance indicators include response times, road accident rates, and feedback from&nbsp;...

✓ Da

In [6]:
# ── RAG SEARCH ──────────────────────────────────────────────────

def search_data_store(query: str, num_results: int = 5) -> dict:
    """
    Query the ADS data store using Discovery Engine.
    Returns both a grounded RAG summary and raw document snippets.
    """
    client = discoveryengine.SearchServiceClient()
    serving_config = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"
        f"/dataStores/{DATA_STORE_ID}/servingConfigs/default_config"
    )

    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=num_results,
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
                return_snippet=True,
                max_snippet_count=3,
            ),
            summary_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec(
                summary_result_count=num_results,
                include_citations=True,
                ignore_adversarial_query=True,
                ignore_non_summary_seeking_query=False,
            ),
        ),
    )

    response = client.search(request)

    # Grounded RAG summary generated by Discovery Engine
    rag_summary = ""
    if response.summary and response.summary.summary_text:
        rag_summary = response.summary.summary_text

    # Supporting document snippets
    documents = []
    for result in response.results:
        doc = result.document
        snippets = []
        title = "Unknown"
        if doc.derived_struct_data:
            for s in doc.derived_struct_data.get("snippets", []):
                if s.get("snippet"):
                    snippets.append(s["snippet"])
            title = doc.derived_struct_data.get("title", "Unknown")
        documents.append({
            "id": doc.id,
            "title": title,
            "snippets": snippets,
        })

    return {
        "rag_summary": rag_summary,
        "documents": documents,
    }


# Smoke test
results = search_data_store("snow removal schedule")
print(f"RAG summary  : {results['rag_summary'][:300] if results['rag_summary'] else 'None'}")
print(f"Documents    : {len(results['documents'])} returned")

RAG summary  : None
Documents    : 5 returned


In [7]:
# ── NATIONAL WEATHER SERVICE API ────────────────────────────────

ALASKA_CITIES = {
    "fairbanks":  (64.8378, -147.7164),
    "anchorage":  (61.2181, -149.9003),
    "juneau":     (58.3005, -134.4197),
    "wasilla":    (61.5814, -149.4411),
    "palmer":     (61.5994, -149.1129),
    "kodiak":     (57.7900, -152.4072),
    "homer":      (59.6425, -151.5483),
    "kenai":      (60.5544, -151.2583),
    "sitka":      (57.0531, -135.3300),
    "ketchikan":  (55.3422, -131.6461),
    "bethel":     (60.7922, -161.7558),
    "nome":       (64.5011, -165.4064),
    "utqiagvik":  (71.2906, -156.7887),
    "barrow":     (71.2906, -156.7887),
    "valdez":     (61.1308, -146.3483),
    "soldotna":   (60.4878, -151.0569),
}

WEATHER_KEYWORDS = [
    "weather", "snow", "forecast", "temperature", "storm", "blizzard",
    "wind", "conditions", "roads", "plowing", "closure", "alert",
    "warning", "ice", "visibility", "freezing",
]


def is_weather_question(text: str) -> bool:
    return any(kw in text.lower() for kw in WEATHER_KEYWORDS)


def extract_city(text: str) -> str | None:
    text_lower = text.lower()
    for city in ALASKA_CITIES:
        if city in text_lower:
            return city
    return None


def get_weather(city: str) -> dict:
    """Fetch current NWS forecast for an Alaska city."""
    lat, lon = ALASKA_CITIES[city]
    headers = {"User-Agent": "ADS-Chatbot/1.0 (alaska-dept-of-snow@alaska.gov)"}

    points_resp = requests.get(
        f"https://api.weather.gov/points/{lat},{lon}",
        headers=headers, timeout=10,
    )
    points_resp.raise_for_status()
    forecast_url = points_resp.json()["properties"]["forecast"]

    forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
    forecast_resp.raise_for_status()
    periods = forecast_resp.json()["properties"]["periods"][:3]

    return {
        "city": city.title(),
        "periods": [
            {
                "name": p["name"],
                "temperature": f"{p['temperature']}°{p['temperatureUnit']}",
                "wind": f"{p['windSpeed']} {p['windDirection']}",
                "forecast": p["detailedForecast"],
            }
            for p in periods
        ],
    }


def get_weather_for_question(question: str) -> str | None:
    """Return formatted weather string if question is weather-related."""
    if not is_weather_question(question):
        return None
    city = extract_city(question) or "fairbanks"
    try:
        data = get_weather(city)
        lines = [f"Current NWS weather for {data['city']}, AK:"]
        for p in data["periods"]:
            lines.append(f"\n{p['name']}: {p['temperature']}, Wind: {p['wind']}")
            lines.append(p["forecast"])
        return "\n".join(lines)
    except Exception as e:
        return f"Weather data temporarily unavailable for {city.title()}, AK."


# Smoke test
print(is_weather_question("What is the snow forecast for Fairbanks?"))  # True
print(is_weather_question("How do I report an unplowed road?"))          # False
print(get_weather_for_question("What is the weather in Anchorage?"))

True
False
Current NWS weather for Anchorage, AK:

Today: 64°F, Wind: 5 to 10 mph SW
A slight chance of rain showers after 4pm. Mostly sunny, with a high near 64. Southwest wind 5 to 10 mph. Chance of precipitation is 20%.

Tonight: 44°F, Wind: 5 to 10 mph SW
A slight chance of rain showers before 10pm. Partly cloudy, with a low around 44. Southwest wind 5 to 10 mph. Chance of precipitation is 20%.

Wednesday: 64°F, Wind: 5 to 15 mph W
Mostly sunny, with a high near 64. West wind 5 to 15 mph.


In [8]:
# ── SECURITY & LOGGING ──────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("ADS-Agent")


def log_step(step: str, detail: str = ""):
    """Log a pipeline step to console and Cloud Logging."""
    msg = f"{step}" + (f": {detail}" if detail else "")
    logger.info(msg)
    print(f"[LOG] {msg}")


# Model Armor setup — fail-open if unavailable
MODEL_ARMOR_ENDPOINT = "modelarmor.us-east4.rep.googleapis.com"
PROMPT_TEMPLATE   = f"projects/{PROJECT_ID}/locations/us-east4/templates/coding-it-prompt-template"
RESPONSE_TEMPLATE = f"projects/{PROJECT_ID}/locations/us-east4/templates/coding-it-response-template"

try:
    from google.cloud import modelarmor_v1
    from google.api_core.client_options import ClientOptions
    model_armor_client = modelarmor_v1.ModelArmorClient(
        client_options=ClientOptions(api_endpoint=MODEL_ARMOR_ENDPOINT)
    )
    MODEL_ARMOR_AVAILABLE = True
    log_step("Model Armor", "Client initialized successfully")
except Exception as e:
    MODEL_ARMOR_AVAILABLE = False
    log_step("Model Armor", f"Unavailable — running fail-open ({e})")


def check_prompt(user_text: str) -> tuple:
    """Run user prompt through Model Armor. Returns (passed, reason)."""
    if not MODEL_ARMOR_AVAILABLE:
        log_step("Model Armor (prompt check)", "SKIPPED — unavailable")
        return True, "ok"
    try:
        request = modelarmor_v1.SanitizeUserPromptRequest(
            name=PROMPT_TEMPLATE,
            user_prompt_data=modelarmor_v1.DataItem(text=user_text),
        )
        response = model_armor_client.sanitize_user_prompt(request=request)
        result = response.sanitization_result
        if result.filter_match_state == modelarmor_v1.FilterMatchState.NO_MATCH_FOUND:
            log_step("Model Armor (prompt check)", "PASSED")
            return True, "ok"
        log_step("Model Armor (prompt check)", "BLOCKED")
        return False, "Blocked by Model Armor prompt filter"
    except Exception as e:
        log_step("Model Armor (prompt check)", f"ERROR — fail-open ({e})")
        return True, "ok"


def check_response(response_text: str) -> tuple:
    """Run model response through Model Armor. Returns (passed, text)."""
    if not MODEL_ARMOR_AVAILABLE:
        log_step("Model Armor (response check)", "SKIPPED — unavailable")
        return True, response_text
    try:
        request = modelarmor_v1.SanitizeModelResponseRequest(
            name=RESPONSE_TEMPLATE,
            model_response_data=modelarmor_v1.DataItem(text=response_text),
        )
        response = model_armor_client.sanitize_model_response(request=request)
        result = response.sanitization_result
        if result.filter_match_state == modelarmor_v1.FilterMatchState.NO_MATCH_FOUND:
            log_step("Model Armor (response check)", "PASSED")
            return True, response_text
        log_step("Model Armor (response check)", "BLOCKED")
        return False, "I'm unable to provide that response."
    except Exception as e:
        log_step("Model Armor (response check)", f"ERROR — fail-open ({e})")
        return True, response_text


# Smoke test
log_step("Security test", "logging is working")
passed, reason = check_prompt("What roads are closed in Fairbanks today?")
print(f"Prompt check — passed: {passed}, reason: {reason}")

[LOG] Model Armor: Client initialized successfully
[LOG] Security test: logging is working
[LOG] Model Armor (prompt check): PASSED
Prompt check — passed: True, reason: ok


In [9]:
# ── AGENT ORCHESTRATION ─────────────────────────────────────────

vertexai.init(project=PROJECT_ID, location=GEMINI_REGION)
model = GenerativeModel(GEMINI_MODEL)

SYSTEM_PROMPT = """You are the Alaska Department of Snow (ADS) virtual assistant.
Your job is to help Alaska residents with questions about snow removal, road conditions,
school closures, and ADS services.

Rules:
1. Only answer questions related to the Alaska Department of Snow and its services.
2. Use information from the provided context when available.
3. If weather data is provided, incorporate it into your answer.
4. If you don't know something, say so — do not make up information.
5. Keep responses concise, clear, and helpful for Alaska residents."""


def ask_ads_agent(user_message: str) -> str:
    """
    Full ADS agent pipeline:
      1. Log incoming message
      2. Model Armor prompt check
      3. Weather API (if relevant)
      4. RAG data store search
      5. Gemini response generation
      6. Model Armor response check
      7. Log and return final response
    """
    print("\n" + "=" * 60)

    # Step 1 — log incoming question
    log_step("User asked question", user_message)

    # Step 2 — Model Armor prompt check
    passed, reason = check_prompt(user_message)
    if not passed:
        log_step("Result", "BLOCKED by Model Armor — response not sent")
        print("=" * 60 + "\n")
        return f"I'm unable to process that request. {reason}"

    # Step 3 — Weather API
    weather_context = ""
    if is_weather_question(user_message):
        log_step("Weather API", "Weather-related question detected — calling NWS")
        weather_context = get_weather_for_question(user_message) or ""
        log_step("Weather API", "Data retrieved" if weather_context else "No data returned")
    else:
        log_step("Weather API", "Not a weather question — skipping")

    # Step 4 — RAG search
    log_step("RAG search", "Querying ADS data store...")
    try:
        rag_results = search_data_store(user_message, num_results=3)
        log_step("RAG search", f"{len(rag_results['documents'])} documents retrieved")
        # Prefer Discovery Engine grounded summary; fall back to raw snippets
        if rag_results["rag_summary"]:
            rag_context = rag_results["rag_summary"]
            log_step("RAG search", "Using grounded summary")
        else:
            rag_context = ""
            for i, doc in enumerate(rag_results["documents"], 1):
                snippets = " ".join(doc.get("snippets", []))
                if snippets:
                    rag_context += f"\nDocument {i} — {doc['title']}:\n{snippets}\n"
            log_step("RAG search", "Using raw document snippets")
    except Exception as e:
        log_step("RAG search", f"ERROR ({e}) — continuing without context")
        rag_context = ""

    # Step 5 — Build Gemini prompt
    prompt_parts = [SYSTEM_PROMPT]
    if rag_context:
        prompt_parts.append(f"\nRelevant ADS Information:\n{rag_context}")
    if weather_context:
        prompt_parts.append(f"\nCurrent NWS Weather Data:\n{weather_context}")
    prompt_parts.append(f"\nCitizen Question: {user_message}\nResponse:")
    full_prompt = "\n".join(prompt_parts)

    # Step 6 — Gemini generation
    log_step("Gemini", "Generating response...")
    try:
        gemini_response = model.generate_content(full_prompt)
        response_text = gemini_response.text.strip()
        log_step("Gemini", "Response generated successfully")
    except Exception as e:
        log_step("Gemini", f"ERROR: {e}")
        print("=" * 60 + "\n")
        return "I'm sorry, I encountered an error. Please try again."

    # Step 7 — Model Armor response check
    passed, final_response = check_response(response_text)
    if not passed:
        log_step("Result", "Response BLOCKED by Model Armor")
        print("=" * 60 + "\n")
        return final_response

    log_step("Result", "Response delivered to user")
    print("=" * 60 + "\n")
    return final_response


# Smoke test
response = ask_ads_agent("What roads are prioritized for snow plowing in Fairbanks?")
print("\nAgent Response:")
print(response)


[LOG] User asked question: What roads are prioritized for snow plowing in Fairbanks?
[LOG] Model Armor (prompt check): PASSED
[LOG] Weather API: Weather-related question detected — calling NWS
[LOG] Weather API: Data retrieved
[LOG] RAG search: Querying ADS data store...
[LOG] RAG search: 3 documents retrieved
[LOG] RAG search: Using raw document snippets
[LOG] Gemini: Generating response...
[LOG] Gemini: Response generated successfully
[LOG] Model Armor (response check): PASSED
[LOG] Result: Response delivered to user


Agent Response:
Based on current NWS weather data for Fairbanks, temperatures are 71°F today and 74°F tomorrow, indicating no snow is present for plowing.

During the winter season, the Alaska Department of Snow (ADS) works with local municipalities to schedule and prioritize plowing routes. ADS resources are dedicated to public roads and infrastructure. Routes to airports and airport access roads are considered high priority.


### Unit Tests

In [10]:
%%ipytest -v

class TestRAGSearch:

    def test_returns_dict(self):
        result = search_data_store("snow removal")
        assert isinstance(result, dict)
        assert "rag_summary" in result
        assert "documents" in result

    def test_returns_documents(self):
        result = search_data_store("road conditions Alaska")
        assert len(result["documents"]) > 0

    def test_documents_have_required_fields(self):
        result = search_data_store("plowing schedule")
        for doc in result["documents"]:
            assert "id" in doc
            assert "title" in doc
            assert "snippets" in doc

    def test_num_results_respected(self):
        result = search_data_store("snow", num_results=2)
        assert len(result["documents"]) <= 2


class TestWeatherAPI:

    def test_known_city_returns_data(self):
        result = get_weather("fairbanks")
        assert "city" in result
        assert "periods" in result
        assert len(result["periods"]) > 0

    def test_period_has_required_fields(self):
        result = get_weather("anchorage")
        period = result["periods"][0]
        assert "name" in period
        assert "temperature" in period
        assert "forecast" in period

    def test_weather_detection_true(self):
        assert is_weather_question("What is the snow forecast for Fairbanks?") == True

    def test_weather_detection_false(self):
        assert is_weather_question("How do I report an unplowed road?") == False

    def test_city_extraction_anchorage(self):
        assert extract_city("Conditions in Anchorage today?") == "anchorage"

    def test_city_extraction_none(self):
        assert extract_city("What is the weather in Paris?") is None


class TestSecurity:

    def test_safe_prompt_passes(self):
        passed, reason = check_prompt("What roads are closed in Fairbanks?")
        assert passed == True

    def test_log_step_does_not_raise(self):
        log_step("Unit test step", "running fine")

    def test_response_check_passes_normal_text(self):
        passed, text = check_response("Road closures are listed on the ADS website.")
        assert passed == True
        assert isinstance(text, str)


class TestAgent:

    def test_returns_string(self):
        response = ask_ads_agent("What is the snow removal schedule?")
        assert isinstance(response, str)
        assert len(response) > 0

    def test_weather_question_handled(self):
        response = ask_ads_agent("What is the current weather in Anchorage?")
        assert isinstance(response, str)
        assert len(response) > 0

    def test_off_topic_redirected(self):
        response = ask_ads_agent("What is the best recipe for apple pie?")
        assert isinstance(response, str)
        assert len(response) > 0

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 16 items

t_eacae56a190f43fa92dce3ae24f81431.py ................                                       [100%]

========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
================================== 16 passed, 1 warning in 10.77s ==================================


In [11]:
# ── EVALUATION ──────────────────────────────────────────────────

eval_questions = [
    "What is the snow removal priority for residential streets?",
    "How do I report an unplowed road?",
    "What are the school closure procedures during a blizzard?",
    "What areas are prioritized for snow plowing?",
]

print("Generating agent responses for evaluation dataset...")
eval_rows = []
for q in eval_questions:
    response = ask_ads_agent(q)
    eval_rows.append({
        "prompt": (
            f"You are the Alaska Department of Snow virtual assistant. "
            f"Answer this question accurately using only ADS information.\n\n"
            f"Question: {q}\nAnswer:"
        ),
        "context": q,
        "instruction": "Answer the citizen's question accurately based on Alaska Department of Snow information.",
    })
    time.sleep(3)

eval_df = pd.DataFrame(eval_rows)
print(f"\nEvaluation dataset: {len(eval_df)} rows")

groundedness_metric = PointwiseMetric(
    metric="groundedness",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={"Groundedness": "The response only uses information relevant to ADS and does not fabricate facts."},
        rating_rubric={"5": "Fully grounded.", "4": "Mostly grounded.", "3": "Somewhat grounded.", "2": "Partially fabricated.", "1": "Mostly fabricated."},
        input_variables=["prompt"],
    )
)

coherence_metric = PointwiseMetric(
    metric="coherence",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={"Coherence": "The response is logically organized and easy for an Alaska resident to understand."},
        rating_rubric={"5": "Completely coherent.", "4": "Mostly coherent.", "3": "Somewhat coherent.", "2": "Mostly unclear.", "1": "Incoherent."},
        input_variables=["prompt"],
    )
)

helpfulness_metric = PointwiseMetric(
    metric="helpfulness",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={"Helpfulness": "The response directly addresses the citizen's question with actionable information."},
        rating_rubric={"5": "Extremely helpful.", "4": "Mostly helpful.", "3": "Somewhat helpful.", "2": "Minimally helpful.", "1": "Not helpful."},
        input_variables=["prompt"],
    )
)

eval_task = EvalTask(
    dataset=eval_df,
    metrics=[groundedness_metric, coherence_metric, helpfulness_metric],
    experiment="ads-agent-evaluation",
)

run_ts = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
result = eval_task.evaluate(
    model=model,
    experiment_run_name=f"ads-agent-eval-{run_ts}",
)

print("\n=== ADS Agent Evaluation Summary ===")
print(result.summary_metrics)

print("\n=== Per-row Results ===")
metric_names = ["groundedness", "coherence", "helpfulness"]
for i, row in result.metrics_table.iterrows():
    prompt_short = str(row.get("prompt", ""))[:80].replace("\n", " ")
    print(f"\n[{i+1}] {prompt_short}...")
    for m in metric_names:
        score = row.get(f"{m}/score", "N/A")
        print(f"     {m:<20} {str(score):>6}")

Generating agent responses for evaluation dataset...

[LOG] User asked question: What is the snow removal priority for residential streets?
[LOG] Model Armor (prompt check): PASSED
[LOG] Weather API: Weather-related question detected — calling NWS
[LOG] Weather API: Data retrieved
[LOG] RAG search: Querying ADS data store...
[LOG] RAG search: 3 documents retrieved
[LOG] RAG search: Using raw document snippets
[LOG] Gemini: Generating response...
[LOG] Gemini: Response generated successfully
[LOG] Model Armor (response check): PASSED
[LOG] Result: Response delivered to user


[LOG] User asked question: How do I report an unplowed road?
[LOG] Model Armor (prompt check): PASSED
[LOG] Weather API: Not a weather question — skipping
[LOG] RAG search: Querying ADS data store...
[LOG] RAG search: 3 documents retrieved
[LOG] RAG search: Using raw document snippets
[LOG] Gemini: Generating response...
[LOG] Gemini: Response generated successfully
[LOG] Model Armor (response check): PASSED
[LOG] 

INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'model_name': 'publishers/google/models/gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Generating a total of 4 responses from Gemini model gemini-2.5-flash.
100%|██████████| 4/4 [00:03<00:00,  1.32it/s]
INFO:vertexai.evaluation._evaluation:All 4 responses are successfully generated from Gemini model gemini-2.5-flash.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 3.041074285000377 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 12 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 12/12 [00:37<00:00,  3.13s/it]
INFO:vertexai.evaluation._evaluation:All 12 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:37.55137325199985 seconds



=== ADS Agent Evaluation Summary ===
{'row_count': 4, 'groundedness/mean': np.float64(4.75), 'groundedness/std': 0.5, 'coherence/mean': np.float64(5.0), 'coherence/std': 0.0, 'helpfulness/mean': np.float64(3.75), 'helpfulness/std': 1.8929694486000912}

=== Per-row Results ===

[1] You are the Alaska Department of Snow virtual assistant. Answer this question ac...
     groundedness            5.0
     coherence               5.0
     helpfulness             5.0

[2] You are the Alaska Department of Snow virtual assistant. Answer this question ac...
     groundedness            5.0
     coherence               5.0
     helpfulness             4.0

[3] You are the Alaska Department of Snow virtual assistant. Answer this question ac...
     groundedness            4.0
     coherence               5.0
     helpfulness             5.0

[4] You are the Alaska Department of Snow virtual assistant. Answer this question ac...
     groundedness            5.0
     coherence               5.0
   